# Batch analysis

Looks over finished batches. Reads only the report written by `hpc finalize` (`load` raises if it is missing); nothing here simulates.
The same figures as files: `uv run python -m moran_graphs.plots <batch>`.
Asymmetric-lung sweeps (several `flow_ratio` values per size) have their own notebook, `notebooks/asymmetric_lungs.ipynb`.

Mean times are conditional on fixation. `rho` counts censored runs (hit `max_steps`) as not fixed; the verification report says if there were any.

In [ ]:
%load_ext autoreload
%autoreload 2  

import matplotlib.pyplot as plt

from moran_graphs.plots import (PROPERTIES, hover_scatter, load, print_batch, print_run_report, property_grid,
                                scatter, versus_random)

## Configuration

The only cell to edit. Several batches are stitched with a `batch` column; a graph in two batches stays two rows.

In [ ]:
BATCHES = ["2026-10-14-respiratory-size-sweep-2"]
R_VALUES = [1.1]  # the per-r figures need exactly one
DIRECTED = [False, True]  # [False] undirected only, [True] directed only

## What ran

Plan, submission, graphs built, then the verification report of each batch.

In [ ]:
for batch in BATCHES:
    print_batch(batch)

### How the batch went

Your wait from submission to the finished report, CPU time and speed, how much worker time went to waste, and what each r cost.

In [ ]:
for batch in BATCHES:
    print_run_report(batch)

## Load

In [ ]:
results, hists = load(BATCHES)
print(f"{results['hash'].nunique()} graphs, r values {sorted(results['r'].unique().tolist())}")
# Random graphs stay whatever DIRECTED is: they are the undirected cloud every type is compared to.
keep = results["r"].isin(R_VALUES) & (results["is_directed"].isin(DIRECTED) | (results["category"] == "Random"))
results = results[keep]
varying = [p for p in PROPERTIES if results[p].dtype != bool and results[p].nunique() > 1]
results.groupby(["category", "is_directed", "r"])[["rho", "steps_mean_given_fixed", "n_censored"]].agg(["count", "mean", "min", "max"])

In [ ]:
results.columns

In [ ]:
results[["batch", "name", "category", "n_nodes", "n_edges", "rho", "rho_lo", "rho_hi",
      "rho_minus_random", "steps_mean_given_fixed", "steps_se_given_fixed",
      "steps_mean_given_fixed_minus_random", "n_random_matches", "n_censored"]].sort_values("rho", ascending=False).head()

## Fixation probability and time

Hover a point to see its graph and recipe.

In [ ]:
results.shape == results[results["steps_mean_given_fixed"] > 0].shape

In [ ]:
n_runs = results["n_fixed"] + results["n_extinct"]
results["steps_mean_absorption"] = (results["n_fixed"] * results["steps_mean_given_fixed"] + results["n_extinct"] * results["steps_mean_given_extinct"]) / n_runs
results["steps_mean_absorption"]

In [ ]:
# Log x: mean times span 154 to 6e7 steps here, so on a linear axis almost every graph sits at 0.
hover_scatter(results, x="steps_mean_given_fixed", y="rho", marker_by="r").update_xaxes(type="log")

In [ ]:
hover_scatter(results, "steps_mean_given_fixed", "steps_sd_given_fixed")

In [ ]:
if results["r"].nunique() > 1:
    hover_scatter(results, "r", "rho").show()

## Against the random cloud, across sizes

Each graph type as a curve over N, against the random graphs with the same N and E (grey: median and interquartile band, 100 per size). Right panel: random mean time over the graph's mean time, so above 1 the graph fixes faster than random; the band is the random IQR over the graph's single value.
A type needs a cloud at every size and one graph per size, the rest is printed. The complete graph has no cloud (no other graph has its N and E); its exact theory is the dashed well-mixed line.
The random graphs are undirected, so for directed types the cloud matches N and E but not the edge directions.

Thick pale red line: the theory formula from `theory.py` (written out in the legend), the anchor the simulated points should sit on within their error bars. Rho and time for the cycle (both directions), directed line and directed star; rho only for the star (no time formula yet) and for the directed single-root trees (line, star, mammalian, fish: only a mutant on the root fixes, so rho = 1/N whatever r).

In [ ]:
animals = ["Mammalian", "Asymmetric lung", "Avian", "Fish"]  # the category labels; the rest are classical graphs

is_animal = results["category"].isin(animals)
is_random = results["category"] == "Random"
for directed in DIRECTED:
    for name, graphs in (("Animal", is_animal), ("Non-animal", ~is_animal & ~is_random)):
        fig = versus_random(results[graphs | is_random], directed)
        fig.suptitle(f"{name} graphs. {fig.get_suptitle()}")

## Structural properties

One panel per property. Look for properties where the respiratory graphs sit outside the random cloud.

In [ ]:
property_grid(results, "rho", varying).suptitle(f"r = {R_VALUES}");

In [ ]:
property_grid(results, "steps_mean_given_fixed", varying).suptitle(f"r = {R_VALUES}");

## Ad hoc

Any two columns of `results`, or a subset of it. `hover=[...]` picks the columns shown on hover (default N, E, r, batch, recipe). `scatter(ax, df, x, y)` draws the same thing on a matplotlib axis.

In [ ]:
hover_scatter(results, "degree_std", "average_shortest_path_length")